In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
data_dir = "./data/temp_dhw/"

train_file = os.path.join(data_dir, "train.npz")
valid_file = os.path.join(data_dir, "val.npz")
test_file = os.path.join(data_dir, "test.npz")

In [3]:
train_data = np.load(train_file, allow_pickle=True)
train_data

NpzFile './data/temp_dhw/train.npz' with keys: x, y

In [4]:
train_data['x'].shape, train_data['y'].shape

((281, 365, 3, 2), (281,))

In [5]:
train_data['y']

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0.,
       1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
       1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
       1., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0.,
       0., 0., 1., 0., 1.

In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
import torch
import torch.nn as nn
import time
import pandas as pd 
import numpy as np
import util
from engine import trainer
import logging


In [8]:
data_dir = "./data/temp_dhw/"
# data_dir = "./data/METR-LA/"
batch_size = 16
shuffle = True
epochs = 20

aptonly = True
addaptadj = True
randomadj = True
adjinit = None

in_dim = 2
seq_length = 365 # 365
num_nodes = 3 #3
nhid = 32
dropout = 0.3
learning_rate = 0.001
weight_decay = 0.0001
gcn_bool = True

device = torch.device('cuda')
print_every = 2
save = './garage.metr'

In [9]:
dataloader = util.load_dataset(data_dir, batch_size, batch_size, batch_size)
scaler = dataloader['scaler']
supports = None


In [10]:
engine = trainer(scaler, in_dim, seq_length, num_nodes, nhid, dropout,
                         learning_rate, weight_decay, device, supports, gcn_bool, addaptadj,
                         adjinit)

In [11]:

for iter, (x, y) in enumerate(dataloader["train_loader"].get_iterator()):
    trainx = torch.Tensor(x).to(device)  # (batch_size, seq_length, num_nodes, in_dim)
    trainy = torch.Tensor(y).to(device)

    trainx = trainx.transpose(1, 3)  # (batch_size, in_dim, seq_length, num_nodes)
    # trainy = trainy.transpose(1, 3)
    # metrics = engine.train(trainx, trainy[:,0,:,:])
    print(f"trainx.shape: {trainx.shape}, trainy.shape: {trainy.shape}")

    engine.model.train()
    engine.optimizer.zero_grad()
    input = nn.functional.pad(trainx, (1, 0, 0, 0))
    print(f"padded input shape: {input.shape}")

    output = engine.model(input)
    print(f"output shape: {output.shape}")

    output = output.transpose(1, 3)
    print(f"output transposed shape: {output.shape}")
    # output = [batch_size,12,num_nodes,1]
    # real = torch.unsqueeze(real_val,dim=1)

    predict = engine.scaler.inverse_transform(output)
    print(f"predict shape: {predict.shape}")

    # loss = engine.loss(predict, real, 0.0)
    # loss.backward()
    # if engine.clip is not None:
    #     torch.nn.utils.clip_grad_norm_(engine.model.parameters(), engine.clip)
    # engine.optimizer.step()
    # mape = util.masked_mape(predict,real,0.0).item()
    # rmse = util.masked_rmse(predict,real,0.0).item()
    break

trainx.shape: torch.Size([16, 2, 3, 365]), trainy.shape: torch.Size([16])
padded input shape: torch.Size([16, 2, 3, 366])
output shape: torch.Size([16, 365, 3, 1])
output transposed shape: torch.Size([16, 1, 3, 365])
predict shape: torch.Size([16, 1, 3, 365])


In [ ]:
# trainx.shape: torch.Size([16, 2, 3, 365]), trainy.shape: torch.Size([16])
# padded input shape: torch.Size([16, 2, 3, 366])
# output shape: torch.Size([16, 365, 3, 354])
# output transposed shape: torch.Size([16, 354, 3, 365])
# predict shape: torch.Size([16, 354, 3, 365])

In [17]:
predict.shape, predict[0, :, 0, :]

(torch.Size([16, 1, 3, 365]),
 tensor([[ 9.1130,  5.9485,  6.5579, 12.1779, 11.9425,  8.1287,  9.2396, 11.8163,
           9.4206,  9.8811, 10.7692,  5.4287, 10.2373, 10.9595, 10.3812,  8.3123,
           6.9958,  7.9747, 10.8377,  7.7468,  4.6206,  6.6690,  7.9156,  7.0810,
           9.3088,  9.3091, 12.8043,  7.3483,  8.2410, 11.3184,  6.6805,  5.8070,
           8.5537,  7.7894,  8.9176, 11.8356,  8.3031,  8.0543,  6.9849,  9.4671,
          10.4318,  8.1832,  8.2630, 11.1651,  8.5253,  8.1791, 11.0004,  8.9082,
           9.0080, 11.6356,  9.2110, 10.9848,  7.5928,  7.6165, 11.2170,  8.7184,
           6.9269, 10.3933,  5.6222,  5.2313, 13.9382, 13.3890,  5.8536, 11.8247,
           8.0113,  9.9295,  7.5120,  3.7892, 10.3330, 11.2167,  8.5819,  7.6917,
          11.6309,  7.8811,  9.9689,  9.6691,  9.3795,  8.5902,  9.5757,  3.8883,
           6.3877, 12.3355,  9.4000, 12.8473, 11.1654,  6.2283,  7.0192,  9.7279,
           9.4262,  6.5560,  5.7269,  9.4283,  8.8657,  8.6783,  8.1

In [24]:
import plotly.express as px

px.line(y=predict[0, :, 0, :].cpu().detach().numpy().flatten())